# Мини-репликация Scaling Monosemanticity на Qwen3-0.6B

В работе Anthropic **Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet** разреженный автоэнкодер (SAE) раскладывает активации языковой модели по большому словарю признаков. Отдельные направления этого словаря часто оказываются заметно понятнее, чем исходные нейроны: они реагируют на темы, синтаксические конструкции, фрагменты кода и более абстрактные свойства текста.

Здесь воспроизводится основной исследовательский маршрут на меньшем масштабе. Мы возьмём активации residual stream из середины `Qwen/Qwen3-0.6B`, обучим на них небольшой SAE, оценим качество реконструкции и разреженность, а затем посмотрим на контексты, которые сильнее всего активируют найденные признаки. При наличии локального vLLM-сервера нескольким признакам можно автоматически предложить текстовые описания.

Это именно мини-репликация, а не попытка получить результаты уровня Claude 3 Sonnet. В ноутбуке нет scaling laws, сравнения словарей на миллионы признаков и экспериментов со steering. Небольшая выборка и короткое обучение особенно в smoke-режиме нужны для проверки всей цепочки, поэтому найденные признаки нельзя считать устойчивыми без отдельного большого запуска.

Все результаты складываются в `artifacts/qwen3_0_6b_sae/`.

## Конфигурация запуска

В одном месте собраны параметры модели, данных, SAE и автоинтерпретации. По умолчанию включён `smoke_test`: он уменьшает выборку, ширину словаря и число шагов, чтобы быстро обнаружить несовместимость модели, cache keys или окружения. Такой прогон проверяет код, но почти ничего не говорит о качестве признаков.

Для содержательного эксперимента нужно выставить `smoke_test=False` и подобрать объём данных и число шагов под доступную GPU. В отличие от статьи, здесь словарь шире residual stream лишь в несколько раз, а не содержит миллионы признаков.

В основном запуске C4 читается потоково: `num_texts` задаёт верхнюю границу числа документов, а `max_activation_vectors` — фактический бюджет токенов для SAE. Обучение проходит по всему собранному набору несколько раз; один проход соответствует одной эпохе.

In [23]:
from __future__ import annotations

import dataclasses
import gc
import html
import importlib
import itertools
import json
import math
import os
import random
import time
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

try:
    import transformer_lens
    from transformer_lens.model_bridge import TransformerBridge
except Exception as exc:
    raise RuntimeError(
        "Для этого ноутбука нужен TransformerLens 3 с TransformerBridge. "
        "Установка: uv pip install 'transformer-lens>=3'"
    ) from exc

print("torch", torch.__version__)
print("transformer_lens", getattr(transformer_lens, "__version__", "unknown"))


torch 2.11.0+cu128
transformer_lens unknown


In [44]:
@dataclasses.dataclass
class ExperimentConfig:
    model_id: str = "Qwen/Qwen3-0.6B"
    dataset_id: str = "allenai/c4"
    dataset_config: str | None = "en"
    dataset_streaming: bool = True
    dataset_split: str = "train"
    text_column: str | None = None
    artifact_dir: str = "artifacts/qwen3_0_6b_sae"
    seed: int = 123
    device: str = "cuda:2"
    dtype: str = "auto"
    smoke_test: bool = False
    num_texts: int = 250_000
    seq_len: int = 256
    activation_batch_size: int = 8
    train_batch_size: int = 2048
    expansion_factor: int = 64
    num_epochs: int = 5
    lr: float = 2e-4
    l1_coeff: float = 5e-3
    metrics_every_steps: int = 100
    checkpoint_every_epochs: int = 1
    max_activation_vectors: int = 8_388_608
    activation_key: str | None = None
    num_features_to_explain: int = 300
    top_k_per_feature: int = 150
    context_window_tokens: int = 64
    vllm_base_url: str = "http://localhost:8000/v1"
    vllm_model: str | None = None
    run_auto_interpretation: bool = True
    run_specificity_scoring: bool = True


CFG = ExperimentConfig()

# Smoke-режим проверяет всю цепочку с минимальными затратами. Для анализа
# признаков его нужно отключить и увеличить объём данных и длительность обучения.
if CFG.smoke_test:
    CFG.num_texts = 32
    CFG.seq_len = 128
    CFG.activation_batch_size = 2
    CFG.train_batch_size = 512
    CFG.expansion_factor = 2
    CFG.num_epochs = 1
    CFG.metrics_every_steps = 2
    CFG.checkpoint_every_epochs = 1
    CFG.max_activation_vectors = 4096
    CFG.num_features_to_explain = 3
    CFG.top_k_per_feature = 8

def choose_device(device: str) -> str:
    # Явное значение важнее автоопределения: так можно закрепить устройство запуска.
    if device != "auto":
        return device
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = choose_device(CFG.device)
random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)

artifact_dir = Path(CFG.artifact_dir)
checkpoint_dir = artifact_dir / "checkpoints"
example_dir = artifact_dir / "feature_examples"
for path in (artifact_dir, checkpoint_dir, example_dir):
    path.mkdir(parents=True, exist_ok=True)

with open(artifact_dir / "config.json", "w") as f:
    json.dump(dataclasses.asdict(CFG) | {"resolved_device": device}, f, indent=2)

print("device:", device)
print("artifact_dir:", artifact_dir.resolve())
print("HF_TOKEN present:", bool(os.environ.get("HF_TOKEN")))


device: cuda:2
artifact_dir: /workspace/workspace/llm-xai/artifacts/qwen3_0_6b_sae
HF_TOKEN present: False


## Загрузка Qwen через TransformerBridge

`TransformerBridge.boot_transformers` сохраняет вычисления исходной Hugging Face-модели и одновременно даёт доступ к промежуточным активациям. Это удобнее, чем вручную расставлять hooks по модулям Qwen и затем следить за изменениями их имён между версиями `transformers`.

Число слоёв и размер residual stream читаются из конфига самой модели. Для SAE берётся слой в середине сети: это не утверждение о том, что он оптимален, а простой и воспроизводимый выбор, близкий к постановке статьи.

In [25]:
bridge = TransformerBridge.boot_transformers(CFG.model_id, device=device)
print(type(bridge))

# У разных версий моста исходная HF-модель лежит под одним из двух имён.
hf_model = getattr(bridge, "model", None) or getattr(bridge, "hf_model", None)
hf_config = getattr(hf_model, "config", None) if hf_model is not None else None
n_layers = getattr(hf_config, "num_hidden_layers", None) or getattr(hf_config, "n_layer", None)
d_model = getattr(hf_config, "hidden_size", None) or getattr(hf_config, "d_model", None)
# Берём начало второй половины блоков; номер слоя сохраняется вместе с артефактами.
middle_layer = int(n_layers // 2) if n_layers is not None else None

print("n_layers:", n_layers)
print("d_model:", d_model)
print("middle_layer:", middle_layer)
if d_model is None:
    raise RuntimeError("Could not infer d_model from the bridged model config.")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

<class 'transformer_lens.model_bridge.bridge.TransformerBridge'>
n_layers: 28
d_model: 1024
middle_layer: 14


## Как выбирается активация

Имена тензоров в кэше зависят от архитектуры и версии TransformerLens. Сначала выполняется один короткий forward pass и печатаются доступные ключи. Затем эвристика ищет трёхмерный тензор с последней размерностью `d_model`, отдавая приоритет residual stream выбранного среднего слоя.

Автоматический выбор обязательно выводится на экран и сохраняется в `activation_cache_meta.json`. Если после обновления библиотек эвристика выбрала не тот тензор, ключ можно явно задать через `CFG.activation_key`.

In [26]:
def cache_items(cache: Any) -> list[tuple[Any, Any]]:
    if hasattr(cache, "items"):
        return list(cache.items())
    if hasattr(cache, "cache_dict"):
        return list(cache.cache_dict.items())
    if isinstance(cache, dict):
        return list(cache.items())
    raise TypeError(f"Unsupported cache type: {type(cache)}")

def tensor_cache_items(cache: Any) -> list[tuple[str, torch.Tensor]]:
    rows = []
    for key, value in cache_items(cache):
        if torch.is_tensor(value):
            rows.append((str(key), value))
    return rows

def get_cache_value(cache: Any, key: str) -> torch.Tensor:
    if hasattr(cache, "__getitem__"):
        try:
            return cache[key]
        except Exception:
            pass
    for candidate_key, value in cache_items(cache):
        if str(candidate_key) == key:
            return value
    raise KeyError(key)

def get_tokenizer():
    for obj in (bridge, getattr(bridge, "model", None), getattr(bridge, "hf_model", None)):
        tok = getattr(obj, "tokenizer", None)
        if tok is not None:
            return tok
    from transformers import AutoTokenizer
    return AutoTokenizer.from_pretrained(CFG.model_id, trust_remote_code=True)

tokenizer = get_tokenizer()

def run_bridge_with_cache(inputs: str | list[str]):
    # Обрезаем текст до forward pass: последующее усечение кэша не экономит память attention.
    texts = [inputs] if isinstance(inputs, str) else inputs
    encoded = tokenizer(
        texts,
        padding=len(texts) > 1,
        truncation=True,
        max_length=CFG.seq_len,
        add_special_tokens=True,
        return_tensors="pt",
    )
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)
    return bridge.run_with_cache(input_ids, attention_mask=attention_mask)

logits, sample_cache = run_bridge_with_cache("The Golden Gate Bridge is in San Francisco.")
tensor_keys = tensor_cache_items(sample_cache)
print(f"Cached tensor keys: {len(tensor_keys)}")
for key, value in tensor_keys[:80]:
    print(f"{key:80s} shape={tuple(value.shape)} dtype={value.dtype}")

def infer_activation_key(cache: Any, layer_idx: int | None, d_model: int) -> str:
    if CFG.activation_key is not None:
        return CFG.activation_key

    items = tensor_cache_items(cache)
    layer_terms = [] if layer_idx is None else [
        f".{layer_idx}.", f"_{layer_idx}_", f"/{layer_idx}/", f"layers.{layer_idx}",
        f"blocks.{layer_idx}", f"layer_{layer_idx}", f".{layer_idx}", f"{layer_idx}."
    ]
    resid_terms = ["resid_post", "residual", "hidden_state", "hidden_states", "resid"]

    # Эвристика сначала фильтрует тензоры по d_model, затем ранжирует имена.
    # Напечатанный список кандидатов оставляем для ручной проверки после обновлений.
    candidates = []
    for key, value in items:
        key_l = key.lower()
        if value.ndim < 2 or value.shape[-1] != d_model:
            continue
        score = 0
        if any(term in key_l for term in resid_terms):
            score += 10
        if layer_terms and any(term in key_l for term in layer_terms):
            score += 5
        if "post" in key_l or "output" in key_l:
            score += 2
        if "attn" in key_l or "mlp" in key_l:
            score -= 2
        candidates.append((score, key, tuple(value.shape)))

    candidates = sorted(candidates, reverse=True)
    print("Activation key candidates:")
    for row in candidates[:20]:
        print(row)
    if not candidates or candidates[0][0] <= 0:
        raise RuntimeError(
            "Could not infer a residual-stream activation key. Set CFG.activation_key "
            "to one of the printed cache keys and rerun from this cell."
        )
    return candidates[0][1]

activation_key = infer_activation_key(sample_cache, middle_layer, int(d_model))
print("Selected activation key:", activation_key)

cache_meta = {
    "model_id": CFG.model_id,
    "n_layers": n_layers,
    "middle_layer": middle_layer,
    "d_model": d_model,
    "activation_key": activation_key,
    "first_tensor_keys": [{"key": key, "shape": list(value.shape), "dtype": str(value.dtype)} for key, value in tensor_keys[:200]],
}
with open(artifact_dir / "activation_cache_meta.json", "w") as f:
    json.dump(cache_meta, f, indent=2)


Cached tensor keys: 1581
embed.hook_in                                                                    shape=(1, 9) dtype=torch.int64
embed.hook_out                                                                   shape=(1, 9, 1024) dtype=torch.float32
hook_embed                                                                       shape=(1, 9, 1024) dtype=torch.float32
rotary_emb.hook_in                                                               shape=(1, 9, 1024) dtype=torch.float32
rotary_emb.hook_cos                                                              shape=(1, 9, 128) dtype=torch.float32
rotary_emb.hook_sin                                                              shape=(1, 9, 128) dtype=torch.float32
blocks.0.hook_in                                                                 shape=(1, 9, 1024) dtype=torch.float32
blocks.0.hook_resid_pre                                                          shape=(1, 9, 1024) dtype=torch.float32
blocks.0.ln1.hook_in     

## Тексты для сбора активаций

Основной запуск читает английскую часть `allenai/c4` в streaming-режиме. Это снимает ограничение прежнего `c4-code-20k`, в котором было всего 20 000 строк, и не требует заранее скачивать весь C4. `CFG.num_texts` ограничивает число просмотренных документов сверху, но сбор останавливается раньше, когда набран `CFG.max_activation_vectors` реальных токенов.

Поток не материализуется целиком в памяти: тексты поступают непосредственно в batches активаций. Если Hugging Face недоступен, остаётся небольшой встроенный корпус только для технической проверки; исследовательские выводы по такому fallback делать нельзя.


In [27]:
def load_texts() -> Iterable[str]:
    try:
        from datasets import load_dataset
        ds = load_dataset(
            CFG.dataset_id,
            CFG.dataset_config,
            split=CFG.dataset_split,
            streaming=CFG.dataset_streaming,
        )
        text_column = CFG.text_column
        for row_index, row in enumerate(itertools.islice(ds, CFG.num_texts)):
            if text_column is None:
                columns = list(row.keys())
                text_column = "text" if "text" in columns else columns[0]
                print(
                    f"Streaming up to {CFG.num_texts} texts from "
                    f"{CFG.dataset_id}[{CFG.dataset_split}], column={text_column!r}"
                )
            yield str(row[text_column])
        return
    except Exception as exc:
        print("Не удалось загрузить датасет; используем встроенный корпус для smoke-теста.")
        print(type(exc).__name__, exc)

    base = [
        "The Golden Gate Bridge connects San Francisco to Marin County.",
        "A Python function can raise an exception when an input has the wrong type.",
        "Neuroscience studies neurons, synapses, memory, perception, and cognition.",
        "The Eiffel Tower, the Tower of Pisa, and the Sistine Chapel are tourist attractions.",
        "A SQL query can join tables, filter rows, and aggregate values.",
        "A train, a ferry, a tunnel, and a bridge are parts of transit infrastructure.",
        "In machine learning, sparse autoencoders can decompose activations into features.",
        "Security vulnerabilities include buffer overflows, injection bugs, and unsafe deserialization.",
    ]
    for index in range(CFG.num_texts):
        yield base[index % len(base)]

texts = load_texts()
texts, preview_texts = itertools.tee(texts)
list(itertools.islice(preview_texts, 3))


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Streaming up to 250000 texts from allenai/c4[train], column='text'


['Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.',
 'Discussion in \'Mac OS X Lion (10.7)\' started by axboi87, Jan 20, 2012.\nI\'ve got a 500gb internal drive and a 240gb SSD.\nWhen trying to restore using disk utility i\'m given the error "Not enough space on disk ____ to restore"\nBut I shoul

## Сбор и нормировка активаций

Для каждого токена сохраняется вектор residual stream и минимальная метаинформация: исходный текст, позиция и строковое представление токена. Эти данные позже связывают численную активацию признака с читаемым контекстом.

В статье входы SAE масштабируются одним коэффициентом так, чтобы средний квадрат L2-нормы был равен размерности residual stream `D`:

\[
\mathbb{E}\left[\lVert x\rVert_2^2\right] = D.
\]

Такая нормировка фиксирует масштаб входа и делает коэффициент L1 осмысленным в рамках конкретного эксперимента. Нулевое заполнение используется только при запасном поштучном проходе; оно помогает собрать batch одинаковой длины, но при серьёзном обучении лучше работать с attention mask и исключать padding из выборки.

In [28]:
def ensure_batch_pos_dmodel(x: torch.Tensor) -> torch.Tensor:
    # Дальше код ожидает единый порядок осей: batch, position, d_model.
    if x.ndim == 2:
        x = x.unsqueeze(0)
    if x.ndim != 3:
        raise ValueError(f"Expected activation tensor with 2 or 3 dims, got {tuple(x.shape)}")
    if x.shape[-1] != d_model and x.shape[0] == d_model:
        x = x.transpose(0, -1)
    return x.detach().float().cpu()

def decode_tokens_for_text(text: str, seq_len: int) -> list[str]:
    encoded = tokenizer(
        text,
        truncation=True,
        max_length=seq_len,
        add_special_tokens=True,
        return_tensors=None,
    )
    ids = encoded.get("input_ids", [])
    if ids and isinstance(ids[0], list):
        ids = ids[0]
    return [tokenizer.decode([tok_id], skip_special_tokens=False) for tok_id in ids]

def chunked(xs: Iterable[str], n: int) -> Iterable[list[str]]:
    batch = []
    for item in xs:
        batch.append(item)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:
        yield batch

activation_blocks = []
metadata_rows = []
collected_vectors = 0
total_batches = math.ceil(CFG.num_texts / CFG.activation_batch_size)

for text_start, batch_texts in enumerate(
    tqdm(chunked(texts, CFG.activation_batch_size), total=total_batches, desc="Collecting activations")
):
    batch_index0 = text_start * CFG.activation_batch_size
    try:
        _, cache = run_bridge_with_cache(batch_texts if len(batch_texts) > 1 else batch_texts[0])
        acts = ensure_batch_pos_dmodel(get_cache_value(cache, activation_key))
        if len(batch_texts) == 1 and acts.shape[0] != 1:
            acts = acts[:1]
    except Exception as exc:
        print("Batched bridge call failed; falling back to per-text calls.")
        print(type(exc).__name__, exc)
        per_text = []
        for text in batch_texts:
            _, cache = run_bridge_with_cache(text)
            per_text.append(ensure_batch_pos_dmodel(get_cache_value(cache, activation_key))[0])
        max_len = min(max(x.shape[0] for x in per_text), CFG.seq_len)
        # Запасной поштучный путь выравнивает длины нулями перед torch.stack.
        # Для большого эксперимента padding лучше исключать через attention mask.
        padded = []
        for x in per_text:
            x = x[:max_len]
            if x.shape[0] < max_len:
                pad = torch.zeros(max_len - x.shape[0], x.shape[-1], dtype=x.dtype)
                x = torch.cat([x, pad], dim=0)
            padded.append(x)
        acts = torch.stack(padded, dim=0)

    acts = acts[:, :CFG.seq_len, :]
    valid_blocks = []
    for local_idx, text in enumerate(batch_texts):
        text_idx = batch_index0 + local_idx
        token_strings = decode_tokens_for_text(text, CFG.seq_len)
        # Берём только реальные токены. Padding из batch не должен обучать SAE.
        pos_count = min(len(token_strings), acts.shape[1])
        valid_blocks.append(acts[local_idx, :pos_count, :])
        for pos in range(pos_count):
            metadata_rows.append({
                "row_id": len(metadata_rows),
                "text_idx": text_idx,
                "token_pos": pos,
                "token": token_strings[pos],
                "text_preview": text[:240],
            })
    block = torch.cat(valid_blocks, dim=0)
    activation_blocks.append(block)
    collected_vectors += block.shape[0]
    # Token budget ограничивает RAM и длительность сбора независимо от размера C4.
    # if collected_vectors >= CFG.max_activation_vectors:
    #     break

raw_acts = torch.cat(activation_blocks, dim=0)[:CFG.max_activation_vectors]
metadata_df = pd.DataFrame(metadata_rows).iloc[:len(raw_acts)].reset_index(drop=True)
metadata_df["row_id"] = np.arange(len(metadata_df))

# Один общий множитель приводит E[||x||^2] к d_model, как в статье.
mean_sq_norm = raw_acts.pow(2).sum(dim=-1).mean().item()
activation_scale = math.sqrt(float(d_model) / max(mean_sq_norm, 1e-12))
acts_norm = raw_acts * activation_scale

print("raw_acts:", tuple(raw_acts.shape))
print("mean_sq_norm before:", mean_sq_norm)
print("activation_scale:", activation_scale)
print("mean_sq_norm after:", acts_norm.pow(2).sum(dim=-1).mean().item())
metadata_df.head()


raw_acts: (8388608, 1024)
mean_sq_norm before: 228114.25
activation_scale: 0.06699984073652979
mean_sq_norm after: 1023.9999389648438


,row_id,text_idx,token_pos,token,text_preview
0,0,0,0,Begin,Beginners BBQ Class Taking Place in Missoula!\...
1,1,0,1,ners,Beginners BBQ Class Taking Place in Missoula!\...
2,2,0,2,BBQ,Beginners BBQ Class Taking Place in Missoula!\...
3,3,0,3,Class,Beginners BBQ Class Taking Place in Missoula!\...
4,4,0,4,Taking,Beginners BBQ Class Taking Place in Missoula!\...


## Разреженный автоэнкодер

Encoder переводит нормированный вектор `x` в неотрицательные активации признаков:

\[
f(x) = \operatorname{ReLU}(W_{enc}(x-b_{dec}) + b_{enc}).
\]

Decoder восстанавливает residual stream как сумму направлений словаря:

\[
\hat{x} = b_{dec} + f(x)W_{dec}.
\]

Функция потерь сочетает MSE реконструкции и L1-штраф на активации. MSE требует сохранить информацию исходного слоя, а L1 заставляет использовать для одного токена сравнительно мало признаков. Строки decoder после каждого шага снова приводятся к единичной норме, иначе модель могла бы искусственно уменьшать L1, увеличивая decoder weights и уменьшая активации encoder.

Коэффициент `l1_coeff` задаёт компромисс: слишком малое значение даёт плотные и плохо разделённые признаки, слишком большое ухудшает реконструкцию. Значение в ноутбуке подходит как отправная точка, но не считается оптимальным для Qwen3-0.6B.

Вместо фиксированного числа случайных шагов используется обычное обучение по эпохам. В начале каждой эпохи порядок активаций перемешивается, после чего каждый вектор встречается ровно один раз. Это делает объём обучения явным: `num_epochs × число собранных токенов`.

In [29]:
class SparseAutoencoder(nn.Module):
    def __init__(self, d_in: int, n_features: int):
        super().__init__()
        self.d_in = d_in
        self.n_features = n_features
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.encoder = nn.Linear(d_in, n_features)
        self.W_dec = nn.Parameter(torch.empty(n_features, d_in))
        nn.init.kaiming_uniform_(self.encoder.weight, a=math.sqrt(5))
        nn.init.zeros_(self.encoder.bias)
        nn.init.normal_(self.W_dec, std=1.0 / math.sqrt(d_in))
        self.normalize_decoder_weights()

    @torch.no_grad()
    def normalize_decoder_weights(self):
        # Фиксированная норма decoder не позволяет обойти L1 простым рескейлингом.
        self.W_dec.div_(self.W_dec.norm(dim=1, keepdim=True).clamp_min(1e-8))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.encoder(x - self.b_dec))

    def forward(self, x: torch.Tensor):
        hidden = self.encode(x)
        recon = self.b_dec + hidden @ self.W_dec
        decoder_norms = self.W_dec.norm(dim=1)
        feature_acts = hidden * decoder_norms
        return recon, hidden, feature_acts


n_features = int(d_model) * CFG.expansion_factor
sae = SparseAutoencoder(int(d_model), n_features).to(device)
optimizer = torch.optim.AdamW(sae.parameters(), lr=CFG.lr, betas=(0.9, 0.99), weight_decay=0.0)
train_acts = acts_norm.to(device)

def sae_loss(x: torch.Tensor):
    # MSE отвечает за реконструкцию, L1 — за редкое использование словаря.
    recon, hidden, feature_acts = sae(x)
    mse = F.mse_loss(recon, x)
    l1 = feature_acts.sum(dim=-1).mean()
    loss = mse + CFG.l1_coeff * l1
    l0 = (hidden > 0).float().sum(dim=-1).mean()
    return loss, {"mse": mse.detach(), "l1": l1.detach(), "mean_l0": l0.detach()}

metrics = []
start = time.time()
global_step = 0
steps_per_epoch = math.ceil(len(train_acts) / CFG.train_batch_size)

for epoch in range(1, CFG.num_epochs + 1):
    permutation = torch.randperm(len(train_acts), device=device)
    progress = tqdm(
        range(0, len(train_acts), CFG.train_batch_size),
        total=steps_per_epoch,
        desc=f"Training SAE: epoch {epoch}/{CFG.num_epochs}",
    )
    for batch_start in progress:
        batch_idx = permutation[batch_start:batch_start + CFG.train_batch_size]
        batch = train_acts[batch_idx]
        optimizer.zero_grad(set_to_none=True)
        loss, parts = sae_loss(batch)
        loss.backward()
        optimizer.step()
        sae.normalize_decoder_weights()
        global_step += 1

        is_last_step = epoch == CFG.num_epochs and global_step == CFG.num_epochs * steps_per_epoch
        if global_step == 1 or global_step % CFG.metrics_every_steps == 0 or is_last_step:
            metrics.append({
                "epoch": epoch,
                "step": global_step,
                "loss": float(loss.detach().cpu()),
                "mse": float(parts["mse"].cpu()),
                "l1": float(parts["l1"].cpu()),
                "mean_l0": float(parts["mean_l0"].cpu()),
                "elapsed_s": time.time() - start,
            })
            progress.set_postfix(loss=f"{float(loss.detach().cpu()):.4f}")

    if epoch % CFG.checkpoint_every_epochs == 0 or epoch == CFG.num_epochs:
        torch.save(
            {
                "cfg": dataclasses.asdict(CFG),
                "epoch": epoch,
                "step": global_step,
                "state_dict": sae.state_dict(),
            },
            checkpoint_dir / f"sae_epoch_{epoch}.pt",
        )

torch.save({"cfg": dataclasses.asdict(CFG), "state_dict": sae.state_dict(), "activation_scale": activation_scale}, artifact_dir / "sae_final.pt")
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(artifact_dir / "training_metrics.csv", index=False)
metrics_df


Training SAE: epoch 1/5:   0%|          | 0/4096 [00:00<?, ?it/s]

Training SAE: epoch 2/5:   0%|          | 0/4096 [00:00<?, ?it/s]

Training SAE: epoch 3/5:   0%|          | 0/4096 [00:00<?, ?it/s]

Training SAE: epoch 4/5:   0%|          | 0/4096 [00:00<?, ?it/s]

Training SAE: epoch 5/5:   0%|          | 0/4096 [00:00<?, ?it/s]

,epoch,step,loss,mse,l1,mean_l0,elapsed_s
0,1,1,22.811756,9.674582,2627.435059,32854.015625,0.273188
1,1,100,2.541165,0.015657,505.101562,240.904785,5.851629
2,1,200,0.858624,0.017173,168.290192,76.030273,11.407058
3,1,300,0.185719,0.020831,32.977581,17.558105,16.927036
4,1,400,0.098862,0.015801,16.612129,7.975098,22.442528
...,...,...,...,...,...,...,...
201,5,20100,0.011392,0.007100,0.858256,0.040527,1133.817233
202,5,20200,0.020881,0.007097,2.756846,0.065430,1139.461521
203,5,20300,0.017578,0.007084,2.098773,0.060059,1145.108213
204,5,20400,0.023101,0.007131,3.194133,0.075195,1150.757813


In [30]:
if not metrics_df.empty:
    display(px.line(metrics_df, x="step", y=["loss", "mse", "l1", "mean_l0"], hover_data=["epoch"], title="Метрики обучения SAE по эпохам"))


## Диагностика словаря

Одной training loss недостаточно, поэтому после обучения считаются четыре простые характеристики:

- `mse_per_element` — средняя ошибка реконструкции одной координаты;
- `explained_variance` — доля вариации активаций, сохранённая реконструкцией; чем ближе к 1, тем лучше;
- `mean_l0` — среднее число ненулевых признаков на токен; это фактическая разреженность, а не L1-штраф;
- `dead_feature_fraction` — доля признаков, которые ни разу не сработали на проверенной выборке.

Высокая explained variance сама по себе не означает интерпретируемость, а низкий `L0` можно получить ценой плохой реконструкции. Эти числа имеет смысл читать вместе. В smoke-режиме доля мёртвых признаков особенно нестабильна: выборка слишком мала, чтобы отличить редкий признак от действительно неработающего.

In [38]:
@torch.no_grad()
def compute_feature_stats(x_cpu: torch.Tensor, chunk_size: int = 2048):
    counts = torch.zeros(n_features, dtype=torch.float64)
    max_vals = torch.full((n_features,), -float("inf"), dtype=torch.float64)
    mse_sum = 0.0
    var_sum = 0.0
    n_rows = 0
    l0_sum = 0.0
    sae.eval()
    for start_idx in tqdm(range(0, len(x_cpu), chunk_size), desc="Feature stats"):
        x = x_cpu[start_idx:start_idx+chunk_size].to(device)
        recon, hidden, feature_acts = sae(x)
        # L0 считаем буквально как число положительных ReLU-активаций.
        active = hidden > 0
        counts += active.sum(dim=0).double().cpu()
        max_vals = torch.maximum(max_vals, feature_acts.max(dim=0).values.double().cpu())
        mse_sum += F.mse_loss(recon, x, reduction="sum").item()
        var_sum += ((x - x.mean(dim=0, keepdim=True)) ** 2).sum().item()
        l0_sum += active.float().sum(dim=-1).sum().item()
        n_rows += x.shape[0]
    freqs = counts / max(n_rows, 1)
    return {
        "freqs": freqs,
        "max_vals": max_vals,
        "mse_per_element": mse_sum / max(n_rows * int(d_model), 1),
        "explained_variance": 1.0 - (mse_sum / max(var_sum, 1e-12)),
        "mean_l0": l0_sum / max(n_rows, 1),
    }

stats = compute_feature_stats(acts_norm)
freq_df = pd.DataFrame({
    "feature_id": np.arange(n_features),
    "frequency": stats["freqs"].numpy(),
    "max_activation": stats["max_vals"].numpy(),
})
freq_df["is_dead"] = freq_df["frequency"] == 0
freq_df.to_csv(artifact_dir / "feature_frequencies.csv", index=False)

summary = {
    "mse_per_element": stats["mse_per_element"],
    "explained_variance": stats["explained_variance"],
    "mean_l0": stats["mean_l0"],
    "dead_feature_fraction": float(freq_df["is_dead"].mean()),
}
with open(artifact_dir / "evaluation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary, freq_df.sort_values("max_activation", ascending=False).head()


Feature stats:   0%|          | 0/4096 [00:00<?, ?it/s]

({'mse_per_element': 0.007091725316058728,
  'explained_variance': 0.9927665930567852,
  'mean_l0': 0.05303144454956055,
  'dead_feature_fraction': 0.5540924072265625},
        feature_id  frequency  max_activation  is_dead
 65346       65346   0.005274       78.747650    False
 56302       56302   0.005274       77.977974    False
 1662         1662   0.005274       71.993156    False
 26010       26010   0.005274       71.461884    False
 59973       59973   0.005274       63.661003    False)

In [39]:
plot_df = freq_df.copy()
plot_df["log10_frequency"] = np.log10(plot_df["frequency"] + 1e-8)
display(px.histogram(plot_df, x="log10_frequency", nbins=80, title="Распределение log10 частот признаков"))


## Контексты максимальной активации

Для первичного просмотра берутся живые признаки с наибольшей наблюдавшейся активацией. Для каждого из них сохраняются токены и тексты, на которых признак сработал сильнее всего. Это не беспристрастная выборка признаков: редкие яркие срабатывания здесь получают преимущество. Зато она быстро показывает, научился ли SAE чему-нибудь связному.

HTML-файлы подсвечивают токены интенсивностью активации. Надёжный признак обычно повторяет похожий паттерн в нескольких независимых контекстах, а не просто реагирует на один документ или частый служебный токен.

Для каждого срабатывания используется симметричное токенное окно: `CFG.context_window_tokens` токенов слева, сам анализируемый токен и столько же токенов справа. В prompt эти части передаются отдельно, поэтому разметчик не теряет позицию, активацию которой он объясняет. У начала и конца текста окно естественно короче.

In [45]:
alive = freq_df.query("frequency > 0").copy()
selected_feature_ids = (
    alive.sort_values("max_activation", ascending=False)["feature_id"]
    .head(CFG.num_features_to_explain)
    .astype(int)
    .tolist()
)
selected_feature_ids


[65346,
 56302,
 1662,
 26010,
 59973,
 33,
 4237,
 49721,
 51329,
 25884,
 49445,
 45577,
 9555,
 10045,
 2084,
 64706,
 63427,
 22910,
 49370,
 15081,
 28957,
 30611,
 15403,
 48375,
 42764,
 32368,
 38221,
 58903,
 36263,
 42581,
 52658,
 46586,
 29518,
 60216,
 53764,
 40339,
 31283,
 48585,
 16980,
 18252,
 10741,
 49210,
 4628,
 34124,
 62013,
 65059,
 1716,
 52109,
 42187,
 293,
 63738,
 25438,
 53758,
 35578,
 45574,
 46034,
 30559,
 39599,
 5623,
 44192,
 52189,
 19247,
 56679,
 30851,
 43696,
 12820,
 3258,
 16766,
 3165,
 14377,
 3909,
 55229,
 65464,
 29471,
 48065,
 24085,
 39260,
 544,
 1159,
 30241,
 62168,
 64139,
 864,
 14861,
 222,
 9481,
 54683,
 24703,
 50843,
 28870,
 41120,
 42458,
 2011,
 14319,
 33810,
 17145,
 42859,
 472,
 27670,
 51829,
 22271,
 7236,
 42519,
 21255,
 41222,
 16396,
 1597,
 8528,
 65055,
 59400,
 43616,
 35976,
 43226,
 30149,
 41156,
 55464,
 47828,
 9799,
 10849,
 31745,
 47372,
 33365,
 33567,
 25442,
 11154,
 50211,
 23739,
 60327,
 15747

In [46]:
@torch.no_grad()
def selected_feature_activations(x_cpu: torch.Tensor, feature_ids: list[int], chunk_size: int = 2048) -> torch.Tensor:
    cols = []
    feature_ids_t = torch.tensor(feature_ids, device=device, dtype=torch.long)
    sae.eval()
    for start_idx in range(0, len(x_cpu), chunk_size):
        x = x_cpu[start_idx:start_idx+chunk_size].to(device)
        _, _, feature_acts = sae(x)
        cols.append(feature_acts.index_select(1, feature_ids_t).cpu())
    return torch.cat(cols, dim=0)

def build_token_window(row_id: int, radius: int) -> dict[str, str]:
    center = metadata_df.iloc[row_id]
    text_rows = metadata_df[metadata_df["text_idx"] == center["text_idx"]].sort_values("token_pos")
    center_pos = int(center["token_pos"])
    left = text_rows[
        (text_rows["token_pos"] >= center_pos - radius)
        & (text_rows["token_pos"] < center_pos)
    ]["token"].tolist()
    right = text_rows[
        (text_rows["token_pos"] > center_pos)
        & (text_rows["token_pos"] <= center_pos + radius)
    ]["token"].tolist()
    left_context = "".join(left)
    activating_token = str(center["token"])
    right_context = "".join(right)
    return {
        "left_context": left_context,
        "activating_token": activating_token,
        "right_context": right_context,
        "token_window": f"{left_context} ⟦{activating_token}⟧ {right_context}",
    }

# Повторно считаем только выбранные столбцы, не удерживая весь [tokens, features] в RAM.
selected_acts = selected_feature_activations(acts_norm, selected_feature_ids)
top_rows = []
for col_idx, feature_id in enumerate(selected_feature_ids):
    vals = selected_acts[:, col_idx]
    k = min(CFG.top_k_per_feature, len(vals))
    top_vals, top_idx = torch.topk(vals, k=k)
    for rank, (value, row_idx) in enumerate(zip(top_vals.tolist(), top_idx.tolist()), start=1):
        meta = metadata_df.iloc[int(row_idx)].to_dict()
        window = build_token_window(int(row_idx), CFG.context_window_tokens)
        top_rows.append({
            "feature_id": feature_id,
            "rank": rank,
            "activation": float(value),
            **meta,
            **window,
        })

top_activations_df = pd.DataFrame(top_rows)
try:
    top_activations_df.to_parquet(artifact_dir / "top_activations.parquet", index=False)
except Exception as exc:
    print("Could not write parquet; writing CSV fallback.")
    print(type(exc).__name__, exc)
    top_activations_df.to_csv(artifact_dir / "top_activations.csv", index=False)

top_activations_df.head(20)


,feature_id,rank,activation,row_id,text_idx,token_pos,token,text_preview,left_context,activating_token,right_context,token_window
0,65346,1,78.747650,6305653,33131,0,Consulta,Consulta las críticas y opiniones de otros cli...,,Consulta,las críticas y opiniones de otros clientes de...,⟦Consulta⟧ las críticas y opiniones de otros...
1,65346,2,78.717018,191215,987,0,ointments,ointments for hemorrhoids » hemorrhoidal cream...,,ointments,for hemorrhoids » hemorrhoidal cream » piles ...,⟦ointments⟧ for hemorrhoids » hemorrhoidal c...
2,65346,3,78.456329,7400046,38921,0,PACE,"PACE Financing Program In San Jacinto, Makes H...",,PACE,"Financing Program In San Jacinto, Makes Home ...","⟦PACE⟧ Financing Program In San Jacinto, Mak..."
3,65346,4,77.756493,8276436,43532,0,saved,saved and made whole in Him.\ntowns that are t...,,saved,and made whole in Him.\ntowns that are too sm...,⟦saved⟧ and made whole in Him.\ntowns that a...
4,65346,5,77.653931,6861521,36078,0,supplier,supplier stone crusher kapasitas 2 ton mesin j...,,supplier,stone crusher kapasitas 2 ton mesin jaw crush...,⟦supplier⟧ stone crusher kapasitas 2 ton mes...
5,65346,6,77.458801,359058,1858,0,evil,evil chef mom: Pulp Muppets!\nMy two favorite ...,,evil,chef mom: Pulp Muppets!\nMy two favorite thin...,⟦evil⟧ chef mom: Pulp Muppets!\nMy two favor...
6,65346,7,77.417114,7771779,40896,0,defense,"defense, foreign relations, immigration, feder...",,defense,", foreign relations, immigration, federal cour...","⟦defense⟧ , foreign relations, immigration, f..."
7,65346,8,77.380257,1312852,6888,0,Owned,"Owned by Bill and Sandy Holbrook, the farm is ...",,Owned,"by Bill and Sandy Holbrook, the farm is locat...","⟦Owned⟧ by Bill and Sandy Holbrook, the farm..."
8,65346,9,77.192253,3604315,18905,0,ECH,ECHS invited applications from eligible candid...,,ECH,S invited applications from eligible candidate...,⟦ECH⟧ S invited applications from eligible ca...
9,65346,10,77.149246,1869788,9804,0,Cog,Coges is a geometric shape of origin with rela...,,Cog,es is a geometric shape of origin with relativ...,⟦Cog⟧ es is a geometric shape of origin with ...


In [47]:
def activation_to_color(value: float, max_value: float) -> str:
    if max_value <= 0:
        return "background-color: transparent"
    alpha = min(max(value / max_value, 0.0), 1.0)
    return f"background-color: rgba(255, 140, 0, {0.12 + 0.78 * alpha:.3f})"

def write_feature_html(feature_id: int):
    examples = top_activations_df[top_activations_df["feature_id"] == feature_id].head(CFG.top_k_per_feature)
    col_idx = selected_feature_ids.index(feature_id)
    max_value = float(examples["activation"].max()) if len(examples) else 0.0
    parts = ["<html><body>", f"<h1>Feature {feature_id}</h1>"]
    for _, example in examples.iterrows():
        text_idx = int(example["text_idx"])
        center_pos = int(example["token_pos"])
        same_text = metadata_df[
            (metadata_df["text_idx"] == text_idx)
            & (metadata_df["token_pos"] >= center_pos - CFG.context_window_tokens)
            & (metadata_df["token_pos"] <= center_pos + CFG.context_window_tokens)
        ].sort_values("token_pos")
        row_ids = same_text["row_id"].astype(int).to_numpy()
        vals = selected_acts[row_ids, col_idx].numpy()
        parts.append(f"<h2>rank {int(example['rank'])}, activation {example['activation']:.4f}</h2>")
        parts.append("<p style='font-family: ui-monospace, SFMono-Regular, Menlo, monospace; line-height: 1.8'>")
        for token_pos, token, value in zip(
            same_text["token_pos"].tolist(), same_text["token"].tolist(), vals.tolist()
        ):
            safe_token = html.escape(token).replace("\n", "<br>")
            style = activation_to_color(float(value), max_value)
            if int(token_pos) == center_pos:
                style += "; outline: 2px solid #b45309; font-weight: 700"
            parts.append(f"<span title='{value:.4f}' style='{style}'>{safe_token}</span>")
        parts.append("</p>")
    parts.append("</body></html>")
    path = example_dir / f"feature_{feature_id}.html"
    path.write_text("\n".join(parts), encoding="utf-8")
    return path

html_paths = [write_feature_html(fid) for fid in selected_feature_ids]
html_paths


[PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_65346.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_56302.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_1662.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_26010.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_59973.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_33.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_4237.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_49721.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_51329.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_25884.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_49445.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_45577.html'),
 PosixPath('artifacts/qwen3_0_6b_sae/feature_examples/feature_9555.html'),
 PosixPath('artifa

## Автоматическое описание признаков через vLLM

Top-activating contexts можно передать отдельной языковой модели и попросить её кратко сформулировать общий паттерн. Для этого в другом терминале запускается OpenAI-совместимый endpoint:

```bash
pip install vllm openai
vllm serve Qwen/Qwen3-0.6B \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name qwen-labeler
```

Модель-разметчик не обязана совпадать с моделью, чьи активации раскладывает SAE. Если `Qwen3-0.6B` даёт нестабильный JSON или слишком общие формулировки, можно поднять более сильную instruct-модель под тем же именем endpoint, например:

```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name qwen-labeler
```

Автоматическое описание — гипотеза для последующей проверки, а не доказательство моносемантичности. Если endpoint не запущен, вычислительная часть ноутбука завершается без ошибки и создаёт пустой файл разметки.

При `CFG.vllm_model=None` ноутбук автоматически использует модель endpoint, если она там одна. Для Qwen thinking отключается: внутреннее рассуждение здесь только расходует лимит токенов и иногда оставляет `message.content` пустым.

In [48]:
def vllm_is_available(base_url: str) -> bool:
    try:
        import urllib.request
        with urllib.request.urlopen(base_url.rstrip("/") + "/models", timeout=2) as response:
            return 200 <= response.status < 300
    except Exception:
        return False

def build_label_prompt(feature_id: int, rows: pd.DataFrame) -> str:
    examples = []
    for _, row in rows.head(CFG.top_k_per_feature).iterrows():
        examples.append({
            "rank": int(row["rank"]),
            "activation": round(float(row["activation"]), 4),
            "left_context": str(row["left_context"]),
            "activating_token": str(row["activating_token"]),
            "right_context": str(row["right_context"]),
        })
    return (
        "You are labeling a sparse autoencoder feature from a language model. "
        "Given top activating token contexts, infer the concise concept represented by the feature. "
        "Return only valid JSON with keys: feature_id, label, description, evidence. "
        "Do not include markdown.\n\n"
        f"feature_id: {feature_id}\n"
        f"top_activating_examples: {json.dumps(examples, ensure_ascii=False)}"
    )

def parse_json_object(text: str | None) -> dict[str, Any]:
    if not text:
        raise ValueError("vLLM вернул пустое message.content")
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.removeprefix("json").strip()
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end >= start:
        text = text[start:end+1]
    return json.loads(text)

labels_path = artifact_dir / "feature_labels.jsonl"
label_records = []

if CFG.run_auto_interpretation and vllm_is_available(CFG.vllm_base_url):
    from openai import OpenAI
    client = OpenAI(base_url=CFG.vllm_base_url, api_key="EMPTY")
    served_models = [model.id for model in client.models.list().data]
    if CFG.vllm_model is not None:
        if CFG.vllm_model not in served_models:
            raise RuntimeError(f"Модель {CFG.vllm_model!r} не найдена в endpoint: {served_models}")
        resolved_vllm_model = CFG.vllm_model
    elif len(served_models) == 1:
        resolved_vllm_model = served_models[0]
    else:
        raise RuntimeError(f"Укажите CFG.vllm_model; endpoint вернул модели: {served_models}")
    print("vLLM labeler:", resolved_vllm_model)

    with open(labels_path, "w", encoding="utf-8") as f:
        for feature_id in tqdm(selected_feature_ids, desc="Labeling features"):
            rows = top_activations_df[top_activations_df["feature_id"] == feature_id]
            prompt = build_label_prompt(feature_id, rows)
            response = client.chat.completions.create(
                model=resolved_vllm_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=800,
                response_format={"type": "json_object"},
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            )
            message = response.choices[0].message
            content = message.content
            # Ответ внешней модели сохраняем даже при сломанном JSON: это упрощает разбор.
            try:
                record = parse_json_object(content)
            except Exception:
                record = {"feature_id": feature_id, "label": None, "description": None, "evidence": [], "finish_reason": response.choices[0].finish_reason, "raw_response": message.model_dump()}
            label_records.append(record)
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
else:
    print(f"Автоинтерпретация пропущена: endpoint {CFG.vllm_base_url} недоступен или опция выключена.")
    labels_path.write_text("", encoding="utf-8")

label_records[:3]


vLLM labeler: Qwen/Qwen3.6-35B-A3B-FP8


Labeling features:   0%|          | 0/300 [00:00<?, ?it/s]

[{'feature_id': 65346,
  'label': 'Product Compatibility and Specifications',
  'description': "This feature activates in contexts describing technical specifications, compatibility lists, and product details for hardware, software, and consumer goods. It is strongly associated with terms indicating fitment (e.g., 'Compatible with', 'Fits'), specific model numbers, and technical attributes of items for sale or review.",
  'evidence': ['Compatible with all Ford B-Max models from 08-08-2012 onwards.',
   'Compatible with ASUS A52J Keyboard is Original/Genuine and new keyboard',
   'Compatible with HP Pavilion dv3-2320ep Keyboard is Original/Genuine',
   'Compatible Epson DFX-5000 black Ribbon Cartridge.',
   'Compatible Brother Toner Cartridge, Black, High Capacity 7,500 page yield.']},
 {'feature_id': 56302,
  'label': 'Commercial Transactions and E-commerce',
  'description': 'This feature activates on tokens related to buying, selling, pricing, and commercial services. It captures con

## Опциональная оценка специфичности

В статье описание признака проверяется не только на самых красивых примерах. Разметчику показывают активирующие контексты и просят оценить, насколько предложенная интерпретация соответствует выделенному месту:

- **0** — описание не связано с контекстом;
- **1** — тема где-то рядом, но связь с активирующим фрагментом слабая;
- **2** — описание в целом подходит к выделенному фрагменту или ближайшему контексту;
- **3** — описание точно указывает на активирующий текст.

Особенно полезно сравнивать сильные и слабые активации: у понятного признака верхняя часть распределения обычно специфичнее. Здесь scoring выключен по умолчанию, поскольку он требует работающего vLLM и добавляет ещё один источник модельной ошибки.

In [49]:
def build_specificity_prompt(label_record: dict[str, Any], rows: pd.DataFrame) -> str:
    examples = []
    for _, row in rows.head(CFG.top_k_per_feature).iterrows():
        examples.append({
            "rank": int(row["rank"]),
            "activation": round(float(row["activation"]), 4),
            "left_context": str(row["left_context"]),
            "activating_token": str(row["activating_token"]),
            "right_context": str(row["right_context"]),
        })
    return (
        "Score how well a sparse-autoencoder feature description matches each activating example. "
        "Use rubric scores 0, 1, 2, or 3. Return only valid JSON with keys feature_id and scores. "
        "Each score item must contain rank, score, and short_reason of at most 12 words. "
        "Return one score for every supplied example.\n\n"
        f"feature_description: {json.dumps(label_record, ensure_ascii=False)}\n"
        f"examples: {json.dumps(examples, ensure_ascii=False)}"
    )

specificity_path = artifact_dir / "feature_specificity_scores.jsonl"
specificity_records = []

if CFG.run_specificity_scoring and label_records and vllm_is_available(CFG.vllm_base_url):
    from openai import OpenAI
    client = OpenAI(base_url=CFG.vllm_base_url, api_key="EMPTY")
    with open(specificity_path, "w", encoding="utf-8") as f:
        for record in tqdm(label_records, desc="Specificity scoring"):
            feature_id = int(record["feature_id"])
            rows = top_activations_df[top_activations_df["feature_id"] == feature_id]
            prompt = build_specificity_prompt(record, rows)
            response = client.chat.completions.create(
                model=resolved_vllm_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2048,
                response_format={"type": "json_object"},
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            )
            message = response.choices[0].message
            content = message.content
            # Невалидный JSON не должен обрывать уже выполненную оценку остальных признаков.
            try:
                score_record = parse_json_object(content)
            except Exception:
                score_record = {"feature_id": feature_id, "scores": [], "finish_reason": response.choices[0].finish_reason, "raw_response": message.model_dump()}
            specificity_records.append(score_record)
            f.write(json.dumps(score_record, ensure_ascii=False) + "\n")
else:
    print("Оценка специфичности пропущена: включите опцию и сначала запустите vLLM.")
    specificity_path.write_text("", encoding="utf-8")

specificity_records[:2]


Specificity scoring:   0%|          | 0/300 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Что сохраняется после запуска

Артефакты нужны, чтобы анализировать результат без повторного сбора активаций и не путать запуски с разными параметрами:

- `config.json` — фактическая конфигурация;
- `activation_cache_meta.json` — слой, размерности и выбранный cache key;
- `sae_final.pt` и `checkpoints/sae_epoch_*.pt` — веса SAE;
- `training_metrics.csv` — динамика loss, MSE, L1 и `L0`;
- `feature_frequencies.csv` — частоты срабатывания признаков;
- `evaluation_summary.json` — сводные метрики реконструкции и разреженности;
- `top_activations.parquet` либо запасной CSV;
- `feature_labels.jsonl` и `feature_specificity_scores.jsonl` — результаты автоинтерпретации;
- `feature_examples/*.html` — подсвеченные контексты.

Веса, кэши и сгенерированные таблицы не следует добавлять в git: они относятся к конкретному запуску и могут быть заметно больше самого ноутбука.

In [ ]:
for path in sorted(artifact_dir.rglob("*")):
    if path.is_file():
        print(path, path.stat().st_size, "bytes")


artifacts/qwen3_0_6b_sae/activation_cache_meta.json 30160 bytes
artifacts/qwen3_0_6b_sae/checkpoints/sae_epoch_1.pt 537140613 bytes
artifacts/qwen3_0_6b_sae/checkpoints/sae_epoch_2.pt 537140613 bytes
artifacts/qwen3_0_6b_sae/checkpoints/sae_epoch_3.pt 537140613 bytes
artifacts/qwen3_0_6b_sae/checkpoints/sae_step_100.pt 268573967 bytes
artifacts/qwen3_0_6b_sae/checkpoints/sae_step_200.pt 268573967 bytes
artifacts/qwen3_0_6b_sae/checkpoints/sae_step_300.pt 268573967 bytes
artifacts/qwen3_0_6b_sae/checkpoints/sae_step_400.pt 268573967 bytes
artifacts/qwen3_0_6b_sae/checkpoints/sae_step_500.pt 268573967 bytes
artifacts/qwen3_0_6b_sae/config.json 862 bytes
artifacts/qwen3_0_6b_sae/evaluation_summary.json 169 bytes
artifacts/qwen3_0_6b_sae/feature_examples/feature_10237.html 276442 bytes
artifacts/qwen3_0_6b_sae/feature_examples/feature_10480.html 392566 bytes
artifacts/qwen3_0_6b_sae/feature_examples/feature_1096.html 545858 bytes
artifacts/qwen3_0_6b_sae/feature_examples/feature_11601.html